# 06b-g — conferma indipendente e scheduled sampling dello STATE

Il braccio scalar del 06b-f era promettente ma secondario. Qui viene prima verificato, congelato, su componenti train mai usati dal 06b-f. Poi cinque continuazioni sincronizzate testano scheduled sampling, orizzonte diretto da 8 ms, scaling e specificità causale.

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})

## 1. Sorgenti immutabili

Sono richiesti dataset composito e artefatti 05t, 06b, 06b-b, 06b-c, 06b-d, 06b-e e 06b-f. Tutti gli artefatti decisionali vengono individuati tramite l'indice SHA-256, anche se Kaggle li monta come `archive.zip`.

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
from src.hayflow_model.causal_voltage_state_coupling_forensic import EXPECTED_06B_INDEX_SHA256
from src.hayflow_model.causal_voltage_bridge_representation_forensic import EXPECTED_06BB_INDEX_SHA256
from src.hayflow_model.nested_coupling_optimization_scaling_forensic import EXPECTED_06BC_INDEX_SHA256
from src.hayflow_model.recursive_voltage_state_contract_forensic import EXPECTED_06BD_INDEX_SHA256
from src.hayflow_model.recursive_joint_repair_matrix import EXPECTED_06BE_INDEX_SHA256
from src.hayflow_model.state_scheduled_sampling_confirmation import EXPECTED_06BF_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
def indexed(name,expected,env):
 override=os.environ.get(env);source=discover_indexed_artifact_source(INPUT_ROOT,expected,override=Path(override) if override else None);assert source is not None,f'Artefatto {name} esatto non trovato.';return source
ARTIFACT_05T_SOURCE=indexed('05t',EXPECTED_05T_INDEX_SHA256,'HAYFLOW_05T_ARTIFACT');ARTIFACT_06B_SOURCE=indexed('06b',EXPECTED_06B_INDEX_SHA256,'HAYFLOW_06B_ARTIFACT');ARTIFACT_06BB_SOURCE=indexed('06b-b',EXPECTED_06BB_INDEX_SHA256,'HAYFLOW_06BB_ARTIFACT');ARTIFACT_06BC_SOURCE=indexed('06b-c',EXPECTED_06BC_INDEX_SHA256,'HAYFLOW_06BC_ARTIFACT');ARTIFACT_06BD_SOURCE=indexed('06b-d',EXPECTED_06BD_INDEX_SHA256,'HAYFLOW_06BD_ARTIFACT')
override_06be=os.environ.get('HAYFLOW_06BE_ARTIFACT');ARTIFACT_06BE_SOURCE=None
for expected in EXPECTED_06BE_INDEX_SHA256:
 candidate=discover_indexed_artifact_source(INPUT_ROOT,expected,override=Path(override_06be) if override_06be else None)
 if candidate is not None:ARTIFACT_06BE_SOURCE=candidate;break
assert ARTIFACT_06BE_SOURCE is not None,'Artefatto 06b-e canonico o confermativo non trovato.'
ARTIFACT_06BF_SOURCE=indexed('06b-f',EXPECTED_06BF_INDEX_SHA256,'HAYFLOW_06BF_ARTIFACT')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06bg_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.'
print({'05t':str(ARTIFACT_05T_SOURCE),'06b':str(ARTIFACT_06B_SOURCE),'06b-b':str(ARTIFACT_06BB_SOURCE),'06b-c':str(ARTIFACT_06BC_SOURCE),'06b-d':str(ARTIFACT_06BD_SOURCE),'06b-e':str(ARTIFACT_06BE_SOURCE),'06b-f':str(ARTIFACT_06BF_SOURCE),'base':str(BASE_SOURCE)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 06b-g][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880;print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Preflight e ruolo di conferma

Il preflight verifica lo ZIP 06b-f membro per membro e costruisce il ruolo di conferma usando componenti train successivi al prefisso già consumato. Un overlap con fit/calibration/development precedenti interrompe l'esecuzione.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import StateScheduledSamplingConfig,StateScheduledSamplingConfirmation
base_cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_recursive_joint_repair_matrix.yml').read_text())['recursive_joint_repair_matrix'];specific=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_state_scheduled_sampling_confirmation.yml').read_text())['state_scheduled_sampling_confirmation'];base_cfg.update(specific);config=StateScheduledSamplingConfig.from_mapping(base_cfg)
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_state_scheduled_sampling_confirmation');assert not OUTPUT_DIR.exists(),f'Output gia presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=StateScheduledSamplingConfirmation(bundle,OUTPUT_DIR,config,ARTIFACT_05T_SOURCE,ARTIFACT_06B_SOURCE,ARTIFACT_06BB_SOURCE,ARTIFACT_06BC_SOURCE,ARTIFACT_06BD_SOURCE,ARTIFACT_06BE_SOURCE,ARTIFACT_06BF_SOURCE,code_revision=REVISION);contract=session.prepare_scheduled_sampling_confirmation();role=contract['independent_confirmation_role']
display({'valid':contract['valid'],'06b-f':contract['source_06bf'],'arms':list(contract['continuation_arms']),'primary':contract['primary_arm'],'fallback':contract['fallback_arm'],'confirmation_trajectories':role['confirmation_trajectory_count'],'prior_overlap':role['previous_role_overlap'],'available_components':role['available_components_by_regime'],'checkpoints':contract['continuation_checkpoints']});assert contract['valid'] and not role['previous_role_overlap'] and contract['confirmation_used_for_checkpoint_selection'] is False

## 3. Continuazioni sincronizzate

I cinque bracci partono dallo stesso checkpoint scalar per seed e condividono minibatch e draw di teacher forcing. Il training non legge mai il ruolo di conferma. Il tracker stampa solo avanzamento, ETA, checkpoint e loss mediana.

In [ ]:
training_report=session.train_synchronized_scheduled_matrix();display({'valid':training_report['valid'],'device':training_report['device'],'same_source':training_report['same_source_checkpoint_within_seed'],'same_stream':training_report['same_window_stream_within_seed'],'same_draws':training_report['same_teacher_forcing_uniform_draws_within_seed'],'confirmation_used':training_report['confirmation_used_during_training']});assert training_report['valid'] and not training_report['confirmation_used_during_training']

In [ ]:
confirmation_report=session.evaluate_independent_confirmation();final_report=session.finalize_scheduled_sampling_confirmation(training_report,confirmation_report)
display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'source_scalar_confirmed':final_report['source_scalar_independently_confirmed'],'primary_passed':final_report['primary_arm_passed'],'fallback_passed':final_report['fallback_arm_passed'],'selected_candidate':final_report['selected_candidate'],'causal_specificity':final_report['causal_specificity_retained'],'06c_authorized':final_report['coupled_06c_canary_authorized'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['full_training_authorized']

## 4. Download stabile

La cella crea lo ZIP in `/kaggle/working`, lo converte in base64 e avvia il download tramite un Blob nel browser.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_state_scheduled_sampling_confirmation','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})